# E1 · Datos de confianza: limpieza, integración y puerta de calidad
**Observatorio de Inteligencia Climática y Gestión de Riesgos Agrícolas · Colima, Tecomán y Manzanillo**

Este cuaderno documenta y justifica cada decisión de preprocesamiento de la capa **bronce → plata → oro**.
El código reutilizable vive en `pipeline/` (así corre solo y lo prueba el CI); aquí se *ejecuta paso a paso* y se explica.

| Sección | Criterio de la rúbrica de Ingeniería de Datos |
|---|---|
| 1-2 | Integración de fuentes (conflictos de esquema, tipos, unidades, nombres y duplicados) |
| 3 | Puerta de calidad automática con cuarentena |
| 4 | Valores faltantes (mecanismo y tratamiento justificado) |
| 5 | Atípicos y ruido |
| 6 | Normalización y estandarización |
| 7 | Discretización y jerarquías conceptuales |
| 8 | Reducción de datos |
| 9 | Capas plata y oro + diccionario de datos |

**Cómo correrlo:** desde la raíz del repo, `pip install -r requirements.txt` y luego *Run All*.

In [ ]:
import sys
from pathlib import Path

RAIZ = Path.cwd().resolve()
while not (RAIZ / "pipeline").exists() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pipeline import config as cfg
from pipeline import integracion as integ
from pipeline import limpieza as lim
from pipeline import oro
from pipeline import puerta_calidad as puerta
from pipeline.run_pipeline import ejecutar

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
plt.rcParams.update({"figure.figsize": (9, 3.6), "axes.grid": True, "grid.alpha": .25})
print("Raíz del repo:", RAIZ)

## 1. Bronce: qué llegó realmente
Los extractores guardan el JSON **tal como lo devuelve la API**, sin tocarlo. Primero se lleva cada archivo a un modelo común
(`fecha, municipio, fuente, tmax_c, tmin_c, precip_mm, viento_max_kmh` + metadatos de linaje).

In [ ]:
crudo = integ.cargar_bronce()
print(f"{len(crudo):,} registros · {crudo['archivo_origen'].nunique()} archivos")
display(crudo.groupby("municipio").agg(
    dias=("fecha", "size"), desde=("fecha", "min"), hasta=("fecha", "max"),
    lat_real=("lat_grid", "first"), lon_real=("lon_grid", "first"), elevacion_m=("elevacion_m", "first")))
crudo.head()

**Hallazgo de integración #1: las coordenadas que pedimos no son las que nos respondieron.** Open-Meteo ajusta cada punto a su
celda de malla. Por eso `lat_grid`/`lon_grid` se guardan en cada registro: es parte del linaje y evita afirmar que el dato es "del centro de Colima"
cuando es de una celda a otra elevación.

In [ ]:
pedidas = {"tecoman": (18.9125, -103.8753), "colima": (19.2433, -103.7250), "manzanillo": (19.0522, -104.3158)}
tab = crudo.groupby("municipio")[["lat_grid", "lon_grid"]].first()
tab["lat_pedida"] = [pedidas[m][0] for m in tab.index]
tab["lon_pedida"] = [pedidas[m][1] for m in tab.index]
# distancia aproximada (1° lat ≈ 111 km; 1° lon ≈ 105 km a esta latitud)
tab["desplazamiento_km"] = np.hypot((tab.lat_grid - tab.lat_pedida) * 111, (tab.lon_grid - tab.lon_pedida) * 105).round(1)
tab

## 2. Integración de fuentes heterogéneas
Conflictos que el módulo `pipeline/integracion.py` resuelve de forma automática:

| Conflicto | Ejemplo | Cómo se resuelve |
|---|---|---|
| **Esquema** | La API usa `temperature_2m_max`; el SMN usa `TMAX` | Mapa de columnas por fuente (`PERFIL_SMN`) hacia un modelo único |
| **Tipos** | Fechas `2026-11-07` vs `07/11/2026`; nulos como `"Nulo"` | `to_datetime` con formato explícito; textos nulos → `NaN` |
| **Unidades** | °F, m/s, pulgadas | Tabla `CONVERSIONES` hacia °C, km/h, mm; una unidad desconocida **detiene** la carga |
| **Zona horaria** | Fechas-hora en UTC | Se convierten a `America/Mexico_City` antes de tomar el día |
| **Nombres** | `Tecomán`, `TECOMAN`, `tecoman_colima` | Normalización sin acentos + alias → `tecoman` (entity resolution) |
| **Duplicados entre capturas** | El mismo día descargado el 8 y el 9 de nov | Se queda la captura más reciente; la otra **va a cuarentena con motivo** |
| **Dos fuentes, mismo día** | SMN y Open-Meteo reportan `tmax_c` | Prioridad por variable (`PRIORIDAD_FUENTES`), `n_fuentes` y `dif_<variable>` documentan la discrepancia |

In [ ]:
integrado, duplicados = integ.integrar(crudo)
print(f"Tras integrar: {len(integrado):,} registros · duplicados de captura enviados a cuarentena: {len(duplicados)}")

Con una sola fuente en bronce todavía no hay discrepancias reales entre fuentes que mostrar. Para **demostrar que el adaptador funciona**,
la siguiente celda usa un ejemplo **sintético y claramente marcado** (no es dato real del SMN): una tabla en °F, con fechas `dd/mm/aaaa`,
nombres con acento y un valor `Nulo`. Cuando tengan el CSV real del SMN solo hay que ajustar `PERFIL_SMN` en `pipeline/config.py`.

In [ ]:
ejemplo_smn = pd.DataFrame({
    "FECHA": ["07/11/2026", "06/11/2026"], "MUNICIPIO": ["Tecomán", "TECOMAN"],
    "TMAX": ["85.0", "Nulo"], "TMIN": ["71.0", "70.0"], "PRECIP": ["0", "1.5"],
})
perfil = {**cfg.PERFIL_SMN, "conversiones": {"tmax_c": "f_a_c", "tmin_c": "f_a_c"}}
adaptado = integ.adaptar_tabla(ejemplo_smn, perfil, archivo="EJEMPLO_SINTETICO.csv")
display(adaptado[["fecha", "municipio", "fuente", "tmax_c", "tmin_c", "precip_mm", "viento_max_kmh"]])

# Consolidación: ¿qué pasa si ambas fuentes reportan el mismo día?
om = integrado.query("municipio == 'tecoman'").tail(2)
consolidado_demo = integ.consolidar_fuentes(pd.concat([om, adaptado], ignore_index=True))
consolidado_demo[["fecha", "municipio", "n_fuentes", "fuentes_usadas", "tmax_c", "dif_tmax_c", "viento_max_kmh"]]

## 3. Puerta de calidad 1 (bronce → plata) con pandera
Un registro **no pasa** si viola alguna regla; **no se borra**: va a `data/cuarentena/` con la regla y el valor que lo rompió.

| Regla | Qué revisa |
|---|---|
| Catálogo | `municipio` y `fuente` existen en la configuración |
| Fecha | Formato válido, entre 2000-01-01 y hoy (sin fechas futuras) |
| Tipos y rangos físicos | Numérico y dentro de rango: tmax [-5, 50], tmin [-5, 45], lluvia [0, 500], viento [0, 200] |
| Regla de negocio | `tmax ≥ tmin` y amplitud ≤ 30 °C |
| Coordenadas | Dentro del estado de Colima |
| Unicidad | Una sola fila por fuente + municipio + fecha |

Un **faltante** (NaN) no es un registro inválido: pasa y se trata en la sección 4.

In [ ]:
aprobados, rechazados, reporte = puerta.aplicar_puerta(integrado)
print("Resultado sobre los datos reales:")
print({k: v for k, v in reporte.items() if k != "rechazos_por_regla"}, reporte["rechazos_por_regla"])

Que los datos reales pasen sin rechazos tiene sentido: Open-Meteo entrega un reanálisis ya consistente. **Eso no prueba que la puerta funcione**,
así que se la prueba con registros dañados a propósito (copia en memoria; no toca ningún archivo):

In [ ]:
dañado = integrado.copy()
dañado.loc[0, "precip_mm"] = -5.0                               # lluvia negativa
dañado.loc[1, ["tmax_c", "tmin_c"]] = [20.0, 25.0]              # máxima menor que mínima
dañado.loc[2, "municipio"] = "mazatlan"                         # fuera de catálogo
dañado.loc[3, "viento_max_kmh"] = 900.0                         # imposible físicamente
dañado.loc[4, "fecha"] = pd.Timestamp("2099-01-01")             # fecha futura
dañado = pd.concat([dañado, integrado.head(2)], ignore_index=True)  # duplicados exactos

ok, cuarentena, rep = puerta.aplicar_puerta(dañado)
print(f"Entraron {rep['registros_recibidos']:,} · aprobados {rep['registros_aprobados']:,} · cuarentena {rep['registros_en_cuarentena']}")
print("Nada se perdió en silencio:", rep["registros_aprobados"] + rep["registros_en_cuarentena"] == rep["registros_recibidos"])
pd.set_option("display.max_colwidth", 120)
cuarentena[["municipio", "fecha", "motivo_rechazo"]]

In [ ]:
pd.Series(rep["rechazos_por_regla"]).sort_values().plot.barh(title="Registros rechazados por regla (prueba con datos dañados)")
plt.xlabel("registros"); plt.tight_layout(); plt.show()

## 4. Valores faltantes
**Dos formas de faltante:** (a) el valor llegó vacío; (b) el *día entero* nunca llegó. `completar_calendario` convierte (b) en filas
con NaN para que sean visibles. Después se diagnostica el mecanismo:

- **MCAR** (completamente al azar): la ausencia no depende de nada.
- **MAR** (al azar condicionado): depende de otras variables observadas (se prueba con Mann-Whitney y chi-cuadrada, con corrección de Bonferroni).
- **MNAR** (no al azar): depende del propio valor faltante; **no se puede probar solo con los datos**, se argumenta por dominio.

In [ ]:
base = lim.completar_calendario(integ.consolidar_fuentes(aprobados))
print("Días ausentes detectados:", int(base["fila_agregada"].sum()))
diag = lim.diagnosticar_faltantes(base)
diag

**Los datos reales no traen faltantes** (el reanálisis es una serie completa). Entonces no hay mecanismo que diagnosticar *sobre estos datos*,
pero el pipeline va a recibir una segunda fuente (estaciones del SMN, que sí tienen huecos) y necesita una estrategia ya justificada.
Para justificarla con evidencia y no por costumbre se hace un **experimento de enmascarado**: se esconden valores reales (10 %), cada método
intenta recuperarlos y se compara el error. Se prueban huecos aislados (un día) y en bloques (5 días seguidos).

In [ ]:
resultados = []
for var in cfg.VARIABLES:
    for esc in ["aleatorio", "bloques"]:
        r = lim.evaluar_imputacion(base, var, escenario=esc)
        r.insert(0, "escenario", esc); r.insert(0, "variable", var)
        resultados.append(r)
exp = pd.concat(resultados, ignore_index=True)
mejor = exp.loc[exp.groupby(["variable", "escenario"])["rmse"].idxmin()][["variable", "escenario", "metodo", "rmse"]]
print("Mejor método por variable y escenario (menor RMSE):")
display(mejor.round(3).reset_index(drop=True))

piv = exp.pivot_table(index=["variable", "escenario"], columns="metodo", values="rmse")
piv.round(2)

In [ ]:
fig, ejes = plt.subplots(1, 4, figsize=(14, 3.4), sharey=False)
for ax, var in zip(ejes, cfg.VARIABLES):
    sub = exp[(exp.variable == var) & (exp.escenario == "aleatorio")].sort_values("rmse")
    ax.barh(sub["metodo"], sub["rmse"]); ax.set_title(var); ax.set_xlabel("RMSE")
plt.suptitle("Error al recuperar valores enmascarados (huecos de 1 día)"); plt.tight_layout(); plt.show()

**Decisión (`imputar`)**, coherente con el experimento:

| Variable | Tratamiento | Por qué |
|---|---|---|
| `tmax_c`, `tmin_c`, `viento_max_kmh` | Interpolación temporal hasta 3 días seguidos; si el hueco es mayor, mediana climatológica del mes/municipio | La interpolación ganó con huecos de 1 día; con bloques de 5 días la ventaja desaparece, por eso el límite de 3 días |
| `precip_mm` | KNN con las demás variables y la estación del año | La lluvia es intermitente: interpolar entre dos días secos o lluviosos no tiene sentido |
| Variable que la fuente **nunca** mide | No se imputa | Inventar viento donde el SMN no mide viento sería fabricar datos |

**Eliminar** filas no se eligió: pierde días de una serie que necesita continuidad para pronosticar en la entrega de Estadística.
Toda imputación queda marcada en `<variable>_imputado`.

In [ ]:
plata = lim.imputar(base)
print("Valores imputados:", {v: int(plata[f"{v}_imputado"].sum()) for v in cfg.VARIABLES})
print("Faltantes restantes:", plata[cfg.VARIABLES].isna().sum().to_dict())

## 5. Atípicos y ruido
**Atípico ≠ error.** Un valor imposible (lluvia negativa) ya fue frenado por la puerta de calidad. Lo que llega aquí es clima extremo, que puede ser
real y es, probablemente, el hallazgo más valioso del observatorio. Por eso la decisión es **marcar y conservar**, nunca borrar.

Tres métodos complementarios:
1. **Z-score robusto** (mediana y MAD) sobre la *anomalía* (valor − mediana de ese mes y municipio), para que un febrero frío no se confunda con un atípico en un clima estacional.
2. **IQR (k = 3)** sobre los días con lluvia: el 40-60 % de los días no llueve, así que un IQR sobre todos los días no sirve.
3. **Isolation Forest** (1 %) para combinaciones raras de variables que ninguna regla univariada ve.

In [ ]:
plata = lim.marcar_atipicos(plata)
cols = [c for c in plata.columns if c.startswith("atipico_")]
resumen = plata.groupby("municipio")[cols].sum()
resumen.loc["TOTAL"] = resumen.sum()
display(resumen)
print(f"{plata['atipico_cualquiera'].mean():.1%} de los días tiene alguna marca (se conservan todos)")

In [ ]:
# ¿Cambia mucho el resultado según el método? IQR clásico (k=1.5) vs z robusto sobre tmin, por municipio
comp = []
for mun, g in plata.groupby("municipio"):
    comp.append({"municipio": mun,
                 "IQR k=1.5 sobre valor crudo": int(lim.atipicos_iqr(g["tmin_c"], 1.5).sum()),
                 "IQR k=3 sobre valor crudo": int(lim.atipicos_iqr(g["tmin_c"], 3).sum()),
                 "z robusto sobre anomalía": int(g["atipico_tmin_c"].sum())})
pd.DataFrame(comp).set_index("municipio")

La comparación muestra por qué se eligió la anomalía: el IQR sobre el valor crudo castiga las noches frías de la temporada de secas
aunque sean normales *para esa época*; el z robusto sobre la anomalía solo señala lo que es raro *para ese mes*.

In [ ]:
g = plata[plata["municipio"] == "manzanillo"]
fig, (a1, a2) = plt.subplots(2, 1, figsize=(11, 5.2), sharex=True)
a1.plot(g["fecha"], g["precip_mm"], lw=.7, color="steelblue")
m = g[g["atipico_precip_mm"]]
a1.scatter(m["fecha"], m["precip_mm"], color="crimson", s=18, zorder=3, label="atípico (IQR k=3)")
a1.set_ylabel("lluvia (mm)"); a1.legend(); a1.set_title("Manzanillo · lluvia y viento con atípicos marcados")
a2.plot(g["fecha"], g["viento_max_kmh"], lw=.7, color="gray")
m = g[g["atipico_viento_max_kmh"]]
a2.scatter(m["fecha"], m["viento_max_kmh"], color="crimson", s=18, zorder=3, label="atípico (z robusto)")
a2.set_ylabel("viento máx (km/h)"); a2.legend(); plt.tight_layout(); plt.show()

Revisión de los extremos: ¿son errores o eventos? Los de mayor puntaje se listan para **contrastarlos con fuentes externas**
(avisos de ciclones del SMN) antes de concluir algo. Un día con lluvia y viento extremos a la vez, en la temporada de ciclones, apunta a un evento real.

In [ ]:
plata.sort_values("z_robusto_viento_max_kmh", ascending=False).head(8)[
    ["municipio", "fecha", "viento_max_kmh", "z_robusto_viento_max_kmh", "precip_mm", "atipico_multivariado"]].round(
    {"viento_max_kmh": 1, "z_robusto_viento_max_kmh": 2, "precip_mm": 1})

**Ruido.** La variación día a día se reduce con media móvil de 7 días (`*_m7`) y lluvia acumulada a 7 días. Es una columna **adicional para
visualizar**: el dato original no se modifica, porque suavizar antes de pronosticar le quitaría información a los modelos de Estadística.

In [ ]:
plata = lim.suavizar_ruido(plata)
g = plata[plata["municipio"] == "colima"].tail(240)
plt.plot(g["fecha"], g["tmax_c"], lw=.6, alpha=.6, label="diario (con ruido)")
plt.plot(g["fecha"], g["tmax_c_m7"], lw=2, label="media móvil 7 días")
plt.title("Colima · temperatura máxima, últimos 240 días"); plt.ylabel("°C"); plt.legend(); plt.tight_layout(); plt.show()

## 6. Normalización y estandarización
- **Min-max**: lleva la variable a [0, 1]. Conserva la forma de la distribución, pero **un solo valor extremo comprime al resto**.
- **Z-score**: media 0 y desviación 1. Es lo que necesitan PCA y MANOVA (entregas posteriores), pero también es sensible a los extremos.

Los parámetros se **guardan** (`params_escalado.json`) para aplicar exactamente la misma transformación a los datos que lleguen mañana.

In [ ]:
params = lim.ajustar_escalado(plata)
esc = lim.aplicar_escalado(plata, params)
display(pd.DataFrame(params).T.round(2))

fig, ejes = plt.subplots(2, 3, figsize=(12, 5))
for fila, v in enumerate(["tmax_c", "precip_mm"]):
    for ax, (col, titulo) in zip(ejes[fila], [(v, "original"), (f"{v}_minmax", "min-max"), (f"{v}_z", "z-score")]):
        ax.hist(esc[col], bins=40, color="slateblue", alpha=.8); ax.set_title(f"{v} · {titulo}")
plt.tight_layout(); plt.show()

In [ ]:
# Efecto de los extremos: la lluvia (cola muy larga) vs la temperatura
pd.DataFrame({
    "mediana min-max": [esc["tmax_c_minmax"].median(), esc["precip_mm_minmax"].median()],
    "p95 min-max": [esc["tmax_c_minmax"].quantile(.95), esc["precip_mm_minmax"].quantile(.95)],
    "max original": [esc["tmax_c"].max(), esc["precip_mm"].max()],
}, index=["tmax_c", "precip_mm"]).round(3)

**Cuándo usar cada una (conclusión con evidencia):** en `precip_mm` el máximo (168.9 mm) empuja casi toda la lluvia hacia valores cercanos a 0
en min-max: casi toda la lluvia queda apretada entre 0 y ~0.12 (un día de 5 mm y uno de 20 mm se ven casi iguales). En `tmax_c` no ocurre. Por eso:
- **Min-max** para variables acotadas y sin colas largas, o para visualizar/comparar contra 0-1 (p. ej. un índice de calor).
- **Z-score** para los análisis multivariados (PCA/MANOVA) y para detectar anomalías relativas.
- Para lluvia, cualquiera de los dos conviene combinarlo con una transformación `log1p` antes de escalar si se usa en modelos sensibles a la escala.

## 7. Discretización y jerarquías conceptuales
Se aplican las tres técnicas clásicas, cada una donde tiene sentido:

| Técnica | Variable | Razón |
|---|---|---|
| **Umbrales de dominio** | `tmax_c` → fresco/cálido/muy_cálido/extremo; `precip_mm` → sin_lluvia/ligera/moderada/fuerte/muy_fuerte | Los cortes significan algo para un productor (35 °C = estrés por calor) |
| **Frecuencia igual** (cuartiles) | `tmin_c` | Garantiza grupos del mismo tamaño aunque la distribución sea asimétrica |
| **Ancho igual** (4 rangos) | `viento_max_kmh` | Referencia simple para comparar contra las otras dos |

In [ ]:
disc = lim.agregar_jerarquias(lim.discretizar(esc))
fig, ejes = plt.subplots(1, 4, figsize=(14, 3.2))
for ax, col in zip(ejes, ["tmax_categoria", "precip_categoria", "tmin_cuartil", "viento_rango"]):
    disc[col].value_counts().sort_index().plot.bar(ax=ax, color="teal"); ax.set_title(col); ax.set_xlabel("")
plt.tight_layout(); plt.show()
pd.crosstab(disc["tmax_categoria"], disc["temporada"], normalize="columns").round(3)

**Jerarquías conceptuales** (permiten agregar en distintos niveles, "roll-up", y bajar al detalle, "drill-down"):

- Tiempo: `fecha → mes → trimestre → temporada (lluvias jun-oct / secas nov-may) → año`
- Lugar: `municipio → zona (Costa/Valle) → estado`

La clasificación de zona (Manzanillo y Tecomán = Costa, Colima = Valle) es una **suposición documentada** del equipo, basada en la elevación
(37 m, 37 m y 495 m según la malla); se puede cambiar en `config.MUNICIPIOS`.

In [ ]:
disc[["fecha", "anio", "trimestre", "mes_nombre", "temporada", "municipio", "zona", "estado"]].drop_duplicates(["municipio", "temporada"]).sort_values(["municipio", "temporada"])

## 8. Reducción de datos
Tres estrategias; se aplican las que aportan algo y se justifica la que no.

1. **Agregación** diario → mensual (se conservan media, **máximo y mínimo**, total y días de lluvia para no perder los extremos).
2. **Selección de variables** por correlación: si dos variables tienen |r| > 0.90 se descarta una.
3. **Muestreo**: *no se aplica*. Son ~3,200 filas; no hay problema de volumen y muestrear rompería la continuidad de la serie.

In [ ]:
mensual = lim.reducir_mensual(disc)
red = 1 - len(mensual) / len(disc)
print(f"Filas diarias: {len(disc):,} → mensuales: {len(mensual):,}  (reducción de {red:.1%})")
print("Lo que se pierde: variación dentro del mes (por eso se conservan máximos, mínimos y la desviación se puede recalcular desde plata).")
mensual.head()

In [ ]:
sel = lim.seleccion_variables(disc)
display(sel["correlaciones"].round(2))
print("Pares redundantes (|r|>0.90):", sel["pares_redundantes"] or "ninguno")
print("Variables conservadas:", sel["conservadas"], "· descartadas:", sel["descartadas"] or "ninguna")

Ninguna variable es redundante (las correlaciones son bajas, |r| ≤ 0.24), así que **no se descarta ninguna**. Es un resultado útil: las cuatro
aportan información distinta y las cuatro entran al análisis multivariado de la entrega E3.

## 9. Capas plata y oro, ejecución automática
Todo lo anterior corre **sin intervención** con un solo comando (`python -m pipeline.run_pipeline`) y lo mismo hace el CI en cada push.

In [ ]:
resultado = ejecutar()
print()
print("Archivos generados:")
for carpeta in [cfg.DIR_PLATA, cfg.DIR_ORO, cfg.DIR_CUARENTENA]:
    for f in sorted(carpeta.glob("*")):
        print(" ", f.relative_to(RAIZ))

In [ ]:
oro.diccionario_df().head(12)

## 10. Resumen de evidencia para la rúbrica

| Criterio | Dónde está la evidencia |
|---|---|
| Valores faltantes: mecanismo y tratamiento justificado | Sección 4 (diagnóstico + experimento de enmascarado) |
| Atípicos y ruido: método justificado y decisión por tipo | Sección 5 |
| Normalización, discretización y reducción con análisis comparativo | Secciones 6, 7 y 8 |
| Integración de fuentes sin duplicados | Secciones 1-2 y `tests/test_pipeline.py` |
| Puerta de calidad con cuarentena y motivo | Sección 3 y `data/cuarentena/` |
| Capas plata y oro automáticas con diccionario | Sección 9 y `data/oro/diccionario_datos.csv` |